# Sequence-based TVT baseline

Bu notebook her kuyuyu ayri bir sequence kaynagi olarak tutar. Train/validation ayrimi satir bazinda degil, kuyu bazinda yapilir. Her pencere son satirinin TVT degerini tahmin eder.

In [1]:
import copy
import random
from typing import Literal

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

from training import (
    RNN_DROP_COLUMNS,
    RNN_FEATURE_COLUMNS,
    RNNTrainingDataset,
    prepare_rnn_features,
)
from utils import (
    get_all_test_ids,
    get_all_training_ids,
    test_folder,
    train_folder,
)

SEED = 42
WINDOW_SIZE = 64
BATCH_SIZE = 256
NUM_EPOCHS = 10
VALID_SIZE = 0.15

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


## Well-level split

Ayni kuyunun pencereleri train ve validation tarafina dagitilmaz. Yarismadaki uc test ID'si de model seciminden ayrilir.

In [2]:
all_training_ids = get_all_training_ids()
reserved_test_ids = set(get_all_test_ids())
modeling_ids = sorted(set(all_training_ids) - reserved_test_ids)

train_well_ids, valid_well_ids = train_test_split(
    modeling_ids,
    test_size=VALID_SIZE,
    random_state=SEED,
    shuffle=True,
)

assert set(train_well_ids).isdisjoint(valid_well_ids)
assert set(train_well_ids).isdisjoint(reserved_test_ids)

print(f"Train wells: {len(train_well_ids)}")
print(f"Validation wells: {len(valid_well_ids)}")
print(f"Reserved test wells: {len(reserved_test_ids)}")

Train wells: 654
Validation wells: 116
Reserved test wells: 3


## Feature preparation

Feature engineering her kuyuda kendi MD sirasi icinde yapilir. `prepare_rnn_features` yalnizca gerekli feature veya TVT degeri eksik olan satirlari kaldirir; kuyular birbirine eklenmez.

In [3]:
train_wells = prepare_rnn_features(
    training_ids=train_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)

valid_wells = prepare_rnn_features(
    training_ids=valid_well_ids,
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
)

test_wells = prepare_rnn_features(
    training_ids=sorted(reserved_test_ids),
    feature_columns=RNN_FEATURE_COLUMNS,
    drop_columns=RNN_DROP_COLUMNS,
    imputed=True,
    data_dir=test_folder,
    target_data_dir=train_folder,
)

len(train_wells), len(valid_wells), len(test_wells)

(654, 116, 3)

## Scaling without leakage

Scaler yalnizca train kuyularinda fit edilir. Validation kuyulari ayni train istatistikleriyle donusturulur. Binary mask kolonlari olceklenmez. TVT de optimize etmeyi kolaylastirmak icin standardize edilir; raporlanan RMSE tekrar gercek TVT birimine cevrilir.

In [4]:
MASK_FEATURE_COLUMNS = [
    "GR_isna",
    "GR_typewell_imputed_mask",
    "GR_neighbor_imputed_mask",
]
CONTINUOUS_FEATURE_COLUMNS = [
    column
    for column in RNN_FEATURE_COLUMNS
    if column not in MASK_FEATURE_COLUMNS
]

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

for well in train_wells.values():
    feature_scaler.partial_fit(
        well.data[CONTINUOUS_FEATURE_COLUMNS]
    )
    target_scaler.partial_fit(well.data[["TVT"]])

target_mean = float(target_scaler.mean_[0])
target_scale = float(target_scaler.scale_[0])

def apply_rnn_scalers(prepared_wells):
    for well in prepared_wells.values():
        well.data.loc[:, CONTINUOUS_FEATURE_COLUMNS] = (
            feature_scaler.transform(
                well.data[CONTINUOUS_FEATURE_COLUMNS]
            )
        )
        well.data.loc[:, "TVT_scaled"] = target_scaler.transform(
            well.data[["TVT"]]
        ).ravel()

apply_rnn_scalers(train_wells)
apply_rnn_scalers(valid_wells)
apply_rnn_scalers(test_wells)

target_mean, target_scale

(11502.671923024185, 637.698604177825)

In [5]:
def make_rnn_datasets(prepared_wells):
    return {
        well_id: RNNTrainingDataset(
            data=well.data,
            window_size=WINDOW_SIZE,
            feature_columns=RNN_FEATURE_COLUMNS,
            target_column="TVT_scaled",
            score_mask_column="is_prediction_row",
        )
        for well_id, well in prepared_wells.items()
        if len(well.data) >= WINDOW_SIZE
    }

train_datasets = make_rnn_datasets(train_wells)
valid_datasets = make_rnn_datasets(valid_wells)
test_datasets = make_rnn_datasets(test_wells)

example_well_id = next(iter(train_datasets))
X_sample, y_sample, score_sample = train_datasets[example_well_id][0]

print(f"Example well: {example_well_id}")
print(f"X shape: {tuple(X_sample.shape)}")
print(f"y shape: {tuple(y_sample.shape)}")
print(f"Prediction row: {bool(score_sample)}")
print(f"Finite X: {bool(torch.isfinite(X_sample).all())}")
print(f"Finite y: {bool(torch.isfinite(y_sample).all())}")

Example well: b38e3116
X shape: (64, 20)
y shape: ()
Prediction row: False
Finite X: True
Finite y: True


## Recurrent model

`nn.RNN`, `nn.GRU` ve `nn.LSTM` ayni arayuzle secilebilir. Recurrent layer tum pencereyi isler. Son timestep'in hidden temsili tek bir TVT tahminine donusturulur. `squeeze(-1)` model ciktisini target ile ayni `(batch,)` sekline getirir ve hatali broadcasting'i engeller.

In [6]:
RNN_TYPES = {
    "rnn": nn.RNN,
    "gru": nn.GRU,
    "lstm": nn.LSTM,
}

class RNNModel(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int = 64,
        output_size: int = 1,
        rnn_type: Literal["rnn", "gru", "lstm"] = "gru",
        num_layers: int = 1,
        dropout: float = 0.2,
    ):
        super().__init__()

        if rnn_type not in RNN_TYPES:
            raise ValueError(f"Invalid RNN type: {rnn_type}")

        recurrent_layer = RNN_TYPES[rnn_type]
        self.rnn = recurrent_layer(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sequence_output, _ = self.rnn(x)
        last_hidden = sequence_output[:, -1, :]
        return self.head(last_hidden).squeeze(-1)

In [7]:
sample_loader = DataLoader(
    train_datasets[example_well_id],
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
X_batch, y_batch, score_mask = next(iter(sample_loader))
X_batch = X_batch.to(device)
y_batch = y_batch.to(device)
score_mask = score_mask.to(device)

model = RNNModel(
    input_size=X_batch.shape[-1],
    hidden_size=64,
    output_size=1,
    rnn_type="gru",
    num_layers=2,
    dropout=0.2,
).to(device)

with torch.no_grad():
    predictions = model(X_batch)

assert predictions.shape == y_batch.shape
predictions.shape, y_batch.shape

(torch.Size([256]), torch.Size([256]))

In [8]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

In [9]:
def train_one_epoch(model, datasets, optimizer):
    model.train()
    squared_error_sum = 0.0
    observation_count = 0

    well_ids = list(datasets)
    random.shuffle(well_ids)

    for well_id in well_ids:
        loader = DataLoader(
            datasets[well_id],
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
        )

        for X_batch, y_batch, score_mask in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            score_mask = score_mask.to(device)

            optimizer.zero_grad(set_to_none=True)
            predictions = model(X_batch)
            loss = loss_fn(predictions, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            if score_mask.any():
                errors_tvt = (predictions - y_batch) * target_scale
                squared_error_sum += torch.sum(
                    errors_tvt[score_mask] ** 2
                ).item()
                observation_count += int(score_mask.sum().item())

    if observation_count == 0:
        raise ValueError("Skorlanacak prediction satiri bulunamadi.")
    return float(np.sqrt(squared_error_sum / observation_count))

@torch.no_grad()
def evaluate(model, datasets):
    model.eval()
    squared_error_sum = 0.0
    observation_count = 0

    for dataset in datasets.values():
        loader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
        )

        for X_batch, y_batch, score_mask in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            score_mask = score_mask.to(device)
            predictions_scaled = model(X_batch)

            predictions_tvt = predictions_scaled * target_scale + target_mean
            targets_tvt = y_batch * target_scale + target_mean
            if score_mask.any():
                squared_error_sum += torch.sum(
                    (predictions_tvt[score_mask] - targets_tvt[score_mask]) ** 2
                ).item()
                observation_count += int(score_mask.sum().item())

    if observation_count == 0:
        raise ValueError("Skorlanacak prediction satiri bulunamadi.")
    return float(np.sqrt(squared_error_sum / observation_count))

## Training

Her epoch'ta kuyularin sirasi karistirilir, fakat bir batch hicbir zaman iki kuyunun pencerelerini karistirmaz. Loss tum satirlardan ogrenilir; raporlanan train ve validation RMSE yalnizca hedef satirinda `TVT_input` bos olan prediction bolgesinde ve gercek TVT biriminde hesaplanir.

In [10]:
history = []
best_valid_rmse = float("inf")
best_model_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    train_score_rmse = train_one_epoch(model, train_datasets, optimizer)
    valid_score_rmse = evaluate(model, valid_datasets)

    history.append(
        {
            "epoch": epoch,
            "train_score_rmse": train_score_rmse,
            "valid_score_rmse": valid_score_rmse,
        }
    )

    if valid_score_rmse < best_valid_rmse:
        best_valid_rmse = valid_score_rmse
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch:02d} | "
        f"train score RMSE: {train_score_rmse:.4f} | "
        f"valid score RMSE: {valid_score_rmse:.4f}"
    )

if best_model_state is not None:
    model.load_state_dict(best_model_state)

test_score_rmse = evaluate(model, test_datasets)
print(f"Reserved test score RMSE: {test_score_rmse:.4f}")

pd.DataFrame(history)

Epoch 01 | train score RMSE: 185.5980 | valid score RMSE: 458.3402
Epoch 02 | train score RMSE: 154.6745 | valid score RMSE: 246.1774
Epoch 03 | train score RMSE: 143.2209 | valid score RMSE: 174.2731
Epoch 04 | train score RMSE: 138.2675 | valid score RMSE: 180.4065
Epoch 05 | train score RMSE: 133.4644 | valid score RMSE: 229.5884
Epoch 06 | train score RMSE: 123.5875 | valid score RMSE: 151.6563
Epoch 07 | train score RMSE: 125.1992 | valid score RMSE: 162.6778
Epoch 08 | train score RMSE: 126.4589 | valid score RMSE: 175.0402
Epoch 09 | train score RMSE: 122.6974 | valid score RMSE: 204.1370
Epoch 10 | train score RMSE: 121.3339 | valid score RMSE: 218.9068
Reserved test score RMSE: 231.1901


,epoch,train_score_rmse,valid_score_rmse
0,1,185.597969,458.340176
1,2,154.674479,246.177425
2,3,143.220893,174.273059
3,4,138.267475,180.406481
4,5,133.464446,229.588412
5,6,123.587502,151.656307
6,7,125.199175,162.677767
7,8,126.458913,175.040161
8,9,122.697449,204.136993
9,10,121.333932,218.906804
